# LAB | Abstractive Question Answering

<div style="border:1px solid #ccc; border-radius:6px; text-align: center;">

<br>

**Run this notebook in [Google Colab](https://colab.research.google.com/) for GPU acceleration.**

<br>

</div>

Abstractive question-answering focuses on the generation of multi-sentence answers to open-ended questions. It usually works by searching massive document stores for relevant information and then using this information to synthetically generate answers. This notebook demonstrates how Pinecone helps you build an abstractive question-answering system. We need three main components:

- A vector index to store and run semantic search
- A retriever model for embedding context passages
- A generator model to generate answers

# Install Dependencies

In [1]:
# We pin exact versions of every library (including torch/torchvision, installed
# from PyTorch's official CUDA 12.1 wheels) so this notebook behaves the same way
# every time it runs, on any machine with a GPU.
!pip install -qU torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu121
!pip install -qU datasets==2.21.0 huggingface_hub==0.25.2 pinecone-client==3.1.0 sentence-transformers==3.0.1 transformers==4.44.2

ERROR: Could not find a version that satisfies the requirement torch==2.4.1 (from versions: 2.5.0+cu121, 2.5.1+cu121)
ERROR: No matching distribution found for torch==2.4.1
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
error: failed-wheel-build-for-install

× Failed to build installable wheels for some pyproject.toml based projects
╰─> tokenizers


# Load and Prepare Dataset

Our source data will be taken from the Wiki Snippets dataset, which contains over 17 million passages from Wikipedia. But, since indexing the entire dataset may take some time, we will only utilize 50,000 passages in this demo that include "History" in the "section title" column. If you want, you may utilize the complete dataset. Pinecone vector database can effortlessly manage millions of documents for you.

In [2]:
from datasets import load_dataset

# We use the "wiki_snippets" dataset, which contains millions of short passages
# extracted from Wikipedia. Each passage comes with useful metadata: the article
# it was taken from (article_title), the section within that article (section_title),
# and the passage's text itself (passage_text). We use the "wiki40b_en_100_0"
# configuration, which contains English passages of ~100 words each.
wiki_data = load_dataset(
    'community-datasets/wiki_snippets',
    name='wiki40b_en_100_0',
    split='train',
    streaming=True
).shuffle(seed=960)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/9.49k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

We are loading the dataset in the streaming mode so that we don't have to wait for the whole dataset to download (which is over 9GB). Instead, we iteratively download records one at a time.

In [3]:
# show the contents of a single document in the dataset
next(iter(wiki_data))

{'_id': '{"datasets_id": 1251236, "wiki_id": "Q2049363", "sp": 16, "sc": 56, "ep": 16, "ec": 109}',
 'datasets_id': 1251236,
 'wiki_id': 'Q2049363',
 'start_paragraph': 16,
 'start_character': 56,
 'end_paragraph': 16,
 'end_character': 109,
 'article_title': 'Panemotichus',
 'section_title': 'Site',
 'passage_text': 'Archaeologists have revealed Iron Age remains there.'}

In [4]:
# filter only documents with History as section_title - Replace None with your code
history = wiki_data.filter(lambda d: d['section_title'].startswith('History'))

Let's iterate through the dataset and apply our filter to select the 50,000 historical passages. We will extract `article_title`, `section_title` and `passage_text` from each document.

In [5]:
from tqdm.auto import tqdm  # progress bar

total_doc_count = 50000

counter = 0
docs = []

# iterate through the dataset and apply our filter
for d in tqdm(history, total=total_doc_count):
    # extract the fields we need - article, section, and passage
    docs.append({
        'article_title': d['article_title'],
        'section_title': d['section_title'],
        'passage_text': d['passage_text']
    })
    # increase the counter on every iteration
    counter += 1
    # stop once we have collected enough passages (the stream is very long)
    if counter == total_doc_count:
        break


  0%|          | 0/50000 [00:00<?, ?it/s]

In [6]:
import pandas as pd

# create a pandas dataframe with the documents we extracted
df = pd.DataFrame(docs)
df.head()

,article_title,section_title,passage_text
0,Paffendorf,History,Paffendorf History Ramon Zenker was involved i...
1,Peranakan,History,the Chinese was widespread.\nIt cannot be deni...
2,Pennsylvania Route 244,History,244 to its original alignment.
3,Peretz (Russian TV channel),History of channel,Peretz (Russian TV channel) History of channel...
4,Patio 29,History,"kidnapped during the 1973 coup.\nIn 1981, Sant..."


# Initialize Pinecone Index

The Pinecone index stores vector representations of our historical passages which we can retrieve later using another vector (query vector). To build our vector index, we must first establish a connection with Pinecone. For this, we need an API from Pinecone. You can get one for free from [here](https://app.pinecone.io/), and after that, we initialize the connection as follows:

In [8]:
!pip install pinecone

import os
import getpass
from pinecone import Pinecone

#
# For this notebook, you'll need a Pinecone API key, 
# to get one, sign up for free at https://app.pinecone.io/,
# then go to "API Keys" and copy the default key (or create a new one).
#
# Once you have your Pinecone API key,
# load it from Colab's "Secrets" manager 
# (the key icon in the left sidebar). 
# Add a secret named PINECONE_API_KEY there and enable notebook access.
#
# Note: Colab Secrets rely on a UI popup to grant access, which only works when
# running in the Colab UI itself. If you're running this notebook through
# another editor (e.g. VS Code's Colab extension), that popup isn't available,
# so we fall back to an environment variable or a secure manual prompt instead.
api_key = os.environ.get('PINECONE_API_KEY')


if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get('PINECONE_API_KEY')
    except Exception:
        api_key = None

if not api_key:
    api_key = getpass.getpass('Enter your Pinecone API key: ')

# configure client
pc = Pinecone(api_key=api_key)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 20.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pinecone]1/2 [pinecone]


Now we setup our index specification, this allows us to define the cloud provider and region where we want to deploy our index. You can find a list of all [available providers and regions here](https://docs.pinecone.io/docs/projects).

In [9]:
from pinecone import ServerlessSpec

cloud = os.environ.get('PINECONE_CLOUD') or 'aws'
region = os.environ.get('PINECONE_REGION') or 'us-east-1'

spec = ServerlessSpec(cloud=cloud, region=region)

Now we create a new index. We will name it "abstractive-question-answering" — you can name it anything we want. We specify the metric type as "cosine" and dimension as 768 because the retriever we use to generate context embeddings is optimized for cosine similarity and outputs 768-dimension vectors.

In [10]:
index_name = 'abstractive-question-answering' # choose a meaningful name for your index

In [11]:
import time

# check if index already exists (it shouldn't if this is first time)
if index_name not in pc.list_indexes().names():
    # create the index: 768 dimensions to match the MPNet retriever, cosine similarity
    pc.create_index(
        name=index_name,
        dimension=768,
        metric='cosine',
        spec=spec
    )
    # wait for the index to be ready before connecting
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1)

# connect to the index and view its stats (should be all zeros on a fresh index)
index = pc.Index(index_name)
index.describe_index_stats()


DescribeIndexStatsResponse(dimension=768, total_vector_count=0, metric='cosine', namespaces=0)

# Initialize Retriever

Next, we need to initialize our retriever. The retriever will mainly do two things:

- Generate embeddings for all historical passages (context vectors/embeddings)
- Generate embeddings for our questions (query vector/embedding)

The retriever will create embeddings such that the questions and passages that hold the answers to our queries are close to one another in the vector space. We will use a SentenceTransformer model based on Microsoft's MPNet as our retriever. This model performs quite well for comparing the similarity between queries and documents. We can use Cosine Similarity to compute the similarity between query and context vectors generated by this model (Pinecone automatically does this for us).

In [12]:
import torch
from sentence_transformers import SentenceTransformer

# set device to GPU if available
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# load the retriever model from huggingface model hub
retriever = SentenceTransformer('flax-sentence-embeddings/all_datasets_v3_mpnet-base', device=device)
retriever

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/9.85k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/591 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'MPNetModel'})
  (1): Pooling({'embedding_dimension': 768, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)

# Generate Embeddings and Upsert

Next, we need to generate embeddings for the context passages. We will do this in batches to help us more quickly generate embeddings and upload them to the Pinecone index. When passing the documents to Pinecone, we need an id (a unique value), context embedding, and metadata for each document representing context passages in the dataset. The metadata is a dictionary containing data relevant to our embeddings, such as the article title, section title, passage text, etc.

In [13]:
# we will use batches of 64
batch_size = 64

#You will create embedding for the passage_text variable and be use to include the meta data in each batch
for i in tqdm(range(0, len(df), batch_size)):
    # find end of batch
    i_end = min(i + batch_size, len(df))
    # extract batch
    batch = df.iloc[i:i_end]
    # generate embeddings for batch
    emb = retriever.encode(batch['passage_text'].tolist()).tolist()
    # get metadata (article, section, passage) for each record
    meta = batch.to_dict(orient='records')
    # create unique ids
    ids = [f"{idx}" for idx in range(i, i_end)]
    # upsert (id, vector, metadata) tuples to Pinecone
    index.upsert(vectors=list(zip(ids, emb, meta)))
# check that we have all vectors in index
index.describe_index_stats()

  0%|          | 0/782 [00:00<?, ?it/s]

DescribeIndexStatsResponse(dimension=768, total_vector_count=50000, metric='cosine', namespaces=1)

# Initialize Generator

We will use ELI5 BART for the generator which is a Sequence-To-Sequence model trained using the ‘Explain Like I’m 5’ (ELI5) dataset. Sequence-To-Sequence models can take a text sequence as input and produce a different text sequence as output.

The input to the ELI5 BART model is a single string which is a concatenation of the query and the relevant documents providing the context for the answer. The documents are separated by a special token &lt;P>, so the input string will look as follows:

>question: What is a sonic boom? context: &lt;P> A sonic boom is a sound associated with shock waves created when an object travels through the air faster than the speed of sound. &lt;P> Sonic booms generate enormous amounts of sound energy, sounding similar to an explosion or a thunderclap to the human ear. &lt;P> Sonic booms due to large supersonic aircraft can be particularly loud and startling, tend to awaken people, and may cause minor damage to some structures. This led to prohibition of routine supersonic flight overland.

More detail on how the ELI5 dataset was built is available [here](https://arxiv.org/abs/1907.09190) and how ELI5 BART model was trained is available [here](https://yjernite.github.io/lfqa.html).

Let's initialize the BART model using transformers.

In [14]:
from transformers import BartTokenizer, BartForConditionalGeneration

# load bart tokenizer and model from huggingface
tokenizer = BartTokenizer.from_pretrained('vblagoje/bart_lfqa')
generator = BartForConditionalGeneration.from_pretrained('vblagoje/bart_lfqa').to(device)

tokenizer_config.json:   0%|          | 0.00/27.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.32k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

All the components of our abstract QA system are complete and ready to be queried. But first, let's write some helper functions to retrieve context passages from Pinecone index and to format the query in the way the generator expects the input.

In [15]:
def query_pinecone(query, top_k):
    # generate embeddings for the query
    xq = retriever.encode(query).tolist()
    # search pinecone index for context passage with the answer
    xc = index.query(vector=xq, top_k=top_k, include_metadata=True)
    return xc

In [16]:
def format_query(query, context):
    # extract passage_text from Pinecone search result and add the <P> tag
    context = [f"<P> {m['metadata']['passage_text']}" for m in context]
    # concatinate all context passages
    context = " ".join(context)
    # contcatinate the query and context passages
    query = f"question: {query} context: {context}"
    return query

Let's test the helper functions. We will query the Pinecone index function we created earlier with the `query_pinecone` to get context passages and pass them to the `format_query` function.

In [17]:
query = "when was the first electric power system built?"
result = query_pinecone(query, top_k=1)
result

QueryResponse(matches=[ScoredVector(id='161', score=0.608927846, values=[], metadata={'article_title': 'Pennsylvania Public Utility Commission', 'passage_text': 'in the way that the electricity was priced and regulated.', 'section_title': 'History'})], namespace='', usage=Usage(read_units=1, write_units=None), response_info=ResponseInfo(raw_headers={'date': 'Tue, 27 Oct 2026 17:23:19 GMT', 'content-type': 'application/json', 'content-length': '299', 'connection': 'keep-alive', 'x-pinecone-max-indexed-lsn': '782', 'x-pinecone-request-latency-ms': '139', 'x-envoy-upstream-service-time': '79', 'x-pinecone-response-duration-ms': '141', 'grpc-status': '0', 'server': 'envoy'}))

In [18]:
from pprint import pprint

In [19]:
# format the query in the form generator expects the input
query = format_query(query, result["matches"])
pprint(query)

('question: when was the first electric power system built? context: <P> in '
 'the way that the electricity was priced and regulated.')


The output looks great. Now let's write a function to generate answers.

In [20]:
def generate_answer(query):
    # tokenize the query to get input_ids
    inputs = tokenizer([query], max_length=1024, return_tensors="pt").to(device)
    # use generator to predict output ids
    ids = generator.generate(inputs["input_ids"], num_beams=2, min_length=20, max_length=40)
    # use tokenizer to decode the output ids
    answer = tokenizer.batch_decode(ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    return pprint(answer)

In [21]:
generate_answer(query)

('The first electric power system was built in the early 19th century. The '
 'first electric power system was built in the early 20th century. The first '
 'electric power system was built in the 1920s')


As we can see, the generator used the provided context to answer our question. Let's run some more queries.

In [22]:
query = "How was the first wireless message sent?"
context = query_pinecone(query, top_k=5)
query = format_query(query, context["matches"])
generate_answer(query)

('The first wireless message was sent by a telegraph operator using Morse '
 'Code. The first wireless message was sent by a telegraph operator using '
 'Morse Code. The first wireless message was sent by a te')


To confirm that this answer is correct, we can check the contexts used to generate the answer.

In [23]:
for doc in context["matches"]:
    print(doc["metadata"]["passage_text"], end='\n---\n')

telegraphy for use by radio operators using Morse Code on continuous wave (CW) telegraphy. It became especially useful, and even essential, to wireless radio operators on both military and civilian ships at sea before the development of advanced single-sideband telephony in the 1960s. Reason for the codes Radio communications, prior to the advent of landlines and satellites as communication paths and relays, was always subject to unpredictable fade outs caused by weather conditions, amount of emission power in kilowatts at the transmitter, radio frequency of the transmission, type of emission, type of transmitting antenna, signal basewave shape and size, sensitivity
---
the Map Communication Model (MCM) has its roots in information theory developed in the telephone industry before the war began. Mathematician, inventor, and teacher Claude Shannon worked at Bell Labs after completing his Ph.D. at the Massachusetts Institute of Technology in 1940. Shannon applied mathematical theory to i

In this case, the answer looks correct. If we ask a question and no relevant contexts are retrieved, the generator will typically return nonsensical or false answers, like with this question about COVID-19:

In [24]:
query = "where did COVID-19 originate?"
context = query_pinecone(query, top_k=3)
query = format_query(query, context["matches"])
generate_answer(query)

('COVID-19 is a virus that infects people who are infected with the virus. It '
 'is a virus that is spread through the air. The virus is spread through the '
 'air by people who')


In [25]:
for doc in context["matches"]:
    print(doc["metadata"]["passage_text"], end='\n---\n')

in the scientific literature until 1965. The most recent outbreak occurred in 1985, on a farm in the town of Stetsonville, also in Wisconsin. Outbreaks of TME have also occurred in Canada, Finland, Germany, and the former Soviet Union.
---
in the United States. Some Canadian poplars were found to be more resistant to the disease than others, resulting in an increase in the plantings of the resistant hybrid clones. In 1941, it was discovered to have spread to Argentina from a shipment of infected poplars from the United States. Human relevance It is unknown how much money has been spent to try to eradicate the disease.  It does receive federal funding through the US Department of Agriculture but the allocation of the funds is unclear.  The Northern Research Station appears to be the leading team on the
---
smuggled into Iraq through Jordan, hidden in a chip in a printer. Media stir The InfoWorld  article reporting purported cyber warfare activity was picked up by other media sources suc

Let’s finish with a final few questions.

In [26]:
query = "what was the war of currents?"
context = query_pinecone(query, top_k=5)
query = format_query(query, context["matches"])
generate_answer(query)

('The War of Currents is a term used to refer to a series of events in the '
 '19th century that were characterized by a series of events that occurred in '
 'the wake of the Napoleonic')


In [27]:
query = "who was the first person on the moon?"
context = query_pinecone(query, top_k=10)
query = format_query(query, context["matches"])
generate_answer(query)

('The first man to walk on the Moon was Neil Armstrong, who walked on the Moon '
 'in 1969. He was the first man to walk on the Moon, and the first man to walk '
 'on the')


In [28]:
query = "what was NASAs most expensive project?"
context = query_pinecone(query, top_k=3)
query = format_query(query, context["matches"])
generate_answer(query)

('The Space Shuttle was the most expensive project in the history of NASA. It '
 'cost about $10 billion to build.')


<br>

### 🚀 Bonus Challenges

Try one or more of these to practice:

- Ask your own questions and compare answers using different `top_k` values.
- Ask a question the dataset likely can't answer and observe how the model responds.
- Change the `section_title` filter to a topic other than "History" and rebuild a small index.
- Print the retrieved passages next to each generated answer to see how retrieval quality affects the result.
- Swap the retriever or generator for a different model from Hugging Face and compare outputs.

#### Helper: answer a question and show the passages it used

This covers the challenge *"Print the retrieved passages next to each generated answer"*. It is used by the other challenges below.

In [29]:
def ask(question, top_k=5, show_context=True, idx=None, answer_fn=None):
    """Retrieve passages for `question`, generate an answer and optionally print the passages."""
    idx = idx or index
    answer_fn = answer_fn or generate_answer
    xq = retriever.encode(question).tolist()
    ctx = idx.query(vector=xq, top_k=top_k, include_metadata=True)
    print(f"Q: {question}  (top_k={top_k})")
    print("A: ", end="")
    answer_fn(format_query(question, ctx["matches"]))
    if show_context:
        print("Retrieved passages:")
        for i, m in enumerate(ctx["matches"], 1):
            md_ = m["metadata"]
            print(f"  {i}. [score {m['score']:.3f}] {md_['article_title']} / {md_['section_title']}")
            print(f"     {md_['passage_text'][:200]}...")
    print()
    return ctx

#### Challenge 1: ask your own questions with different `top_k` values

More passages give the generator more evidence, but also more noise. Watch how the answer changes as `top_k` grows.

In [30]:
my_questions = [
    "how did the industrial revolution change transportation?",
    "why did the roman empire fall?",
    "when was the printing press invented?",
]

for q in my_questions:
    for k in [1, 3, 10]:
        ask(q, top_k=k, show_context=False)
    print("-" * 80)

Q: how did the industrial revolution change transportation?  (top_k=1)
A: ("I can't speak to the industrial revolution as a whole, but I can speak to "
 'the industrial revolution as a whole. The industrial revolution changed '
 'transportation.')

Q: how did the industrial revolution change transportation?  (top_k=3)
A: ('The Industrial Revolution changed the way we transport goods. It changed the '
 'way we transport people. It changed the way we transport goods. It changed '
 'the way we transport people. It changed the way we')

Q: how did the industrial revolution change transportation?  (top_k=10)
A: ('The Industrial Revolution changed the way that people traveled. It changed '
 'the way that people worked. It changed the way that people lived. It changed '
 'the way that people worked. It changed the way that')

--------------------------------------------------------------------------------
Q: why did the roman empire fall?  (top_k=1)
A: ("I'm going to assume you're talking abo

#### Challenge 2: a question the dataset likely can't answer

Our index only holds "History" sections of Wikipedia (and the dump predates recent events), so modern or non-historical questions retrieve unrelated passages. The low similarity scores are a useful warning sign that the generated answer is probably made up.

In [31]:
unanswerable = [
    "what is the best programming language for machine learning in 2024?",
    "how do I bake sourdough bread?",
]

for q in unanswerable:
    ctx = ask(q, top_k=3)
    best = max(m["score"] for m in ctx["matches"])
    print(f"Best similarity score: {best:.3f} "
          f"({'probably unreliable' if best < 0.5 else 'looks relevant'})\n")

Q: what is the best programming language for machine learning in 2024?  (top_k=3)
A: ('I\'m not sure what you mean by "best programming language for machine '
 'learning in 2024". There are a lot of different types of machine learning, '
 'and there are a lot of different types of machine')
Retrieved passages:
  1. [score 0.366] Prolog / History & Use in industry
     of the form not(Bᵢ) are shown by trying and failing to solve the corresponding positive conditions Bᵢ.
Subsequent extensions of Prolog by the original team introduced constraint logic programming abil...
  2. [score 0.349] Just-in-time compilation / History & Security
     and extensively uses, this research base.
The HP project Dynamo was an experimental JIT compiler where the 'bytecode' format and the machine code format were the same; the system turned PA-6000 machin...
  3. [score 0.346] JSON / History
     programming languages. JSON's website lists JSON libraries by language.
Though JSON was originally advertised an

#### Challenge 3: a different `section_title` filter in a small new index

We build a second, smaller index (5,000 passages) from "Geography" sections, using the same retriever and the same steps as above.

In [32]:
from itertools import islice

geo_index_name = "abstractive-qa-geography"
geo_doc_count = 5000

# filter the stream on a different section title and take the first 5,000 passages
geography = wiki_data.filter(lambda d: d["section_title"].startswith("Geography"))
geo_docs = [
    {"article_title": d["article_title"],
     "section_title": d["section_title"],
     "passage_text": d["passage_text"]}
    for d in tqdm(islice(geography, geo_doc_count), total=geo_doc_count)
]
geo_df = pd.DataFrame(geo_docs)
geo_df.head()

  0%|          | 0/5000 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# create the new index (same settings as before)
if geo_index_name not in pc.list_indexes().names():
    pc.create_index(name=geo_index_name, dimension=768, metric="cosine", spec=spec)
    while not pc.describe_index(geo_index_name).status["ready"]:
        time.sleep(1)
geo_index = pc.Index(geo_index_name)

# embed and upsert in batches
for i in tqdm(range(0, len(geo_df), batch_size)):
    i_end = min(i + batch_size, len(geo_df))
    batch = geo_df.iloc[i:i_end]
    emb = retriever.encode(batch["passage_text"].tolist()).tolist()
    meta = batch.to_dict(orient="records")
    ids = [f"geo-{idx}" for idx in range(i, i_end)]
    geo_index.upsert(vectors=list(zip(ids, emb, meta)))

geo_index.describe_index_stats()

In [ ]:
# questions suited to geography sections
ask("what is the climate of the sahara desert like?", top_k=5, idx=geo_index)
ask("which rivers flow through germany?", top_k=5, idx=geo_index)

# same question against the History index, for comparison
ask("what is the climate of the sahara desert like?", top_k=5, idx=index)

#### Challenge 4: print retrieved passages next to each answer

The `ask` helper above already does this (`show_context=True`). Here it is on the lab's own questions, so you can judge whether each answer is supported by its passages.

In [ ]:
for q in ["when was the first electric power system built?",
          "what was the war of currents?",
          "who was the first person on the moon?"]:
    ask(q, top_k=3, show_context=True)

#### Challenge 5a: swap the generator

We replace ELI5 BART with `google/flan-t5-base`, an instruction-tuned model. It expects a natural-language prompt rather than the `<P>` format, so we rebuild the prompt from the passages before generating.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

t5_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
t5_generator = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base").to(device)

def generate_answer_t5(bart_style_query):
    # convert "question: ... context: <P> ... <P> ..." into an instruction prompt
    question, context = bart_style_query.split(" context: ", 1)
    question = question.replace("question: ", "", 1)
    context = context.replace("<P> ", "\n- ")
    prompt = (f"Answer the question in a few sentences using the context.\n"
              f"Context:{context}\nQuestion: {question}")
    inputs = t5_tokenizer(prompt, max_length=512, truncation=True, return_tensors="pt").to(device)
    ids = t5_generator.generate(**inputs, num_beams=2, min_length=20, max_length=80)
    answer = t5_tokenizer.decode(ids[0], skip_special_tokens=True)
    return pprint(answer)

for q in ["How was the first wireless message sent?",
          "what was the war of currents?"]:
    print("ELI5 BART:")
    ask(q, top_k=5, show_context=False)
    print("FLAN-T5:")
    ask(q, top_k=5, show_context=False, answer_fn=generate_answer_t5)
    print("-" * 80)

#### Challenge 5b: swap the retriever

A different retriever produces vectors of a different size (`all-MiniLM-L6-v2` makes 384-dim vectors), so it can't query our 768-dim Pinecone index. To compare without building another index, we embed a 5,000-passage sample of `df` with both models locally and compare what each one retrieves.

In [ ]:
from sentence_transformers import util

sample = df.sample(5000, random_state=42).reset_index(drop=True)
passages = sample["passage_text"].tolist()

retrievers = {
    "mpnet (768-d)": retriever,
    "MiniLM (384-d)": SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device),
}
corpus_emb = {name: m.encode(passages, convert_to_tensor=True, show_progress_bar=True)
              for name, m in retrievers.items()}

for q in ["when was the first electric power system built?",
          "why did the roman empire fall?"]:
    print(f"Q: {q}")
    for name, m in retrievers.items():
        hits = util.semantic_search(m.encode(q, convert_to_tensor=True), corpus_emb[name], top_k=3)[0]
        print(f"  {name}:")
        for h in hits:
            row = sample.iloc[h["corpus_id"]]
            print(f"    [{h['score']:.3f}] {row['article_title']}: {row['passage_text'][:120]}...")
    print()

#### Clean up the bonus index

Delete the Geography index too when you're done (the main index is deleted in the cell below).

In [ ]:
pc.delete_index(geo_index_name)

<br>

---

### Clean Up

Now that we're done experimenting, let's delete the Pinecone index we created. This frees up the resources it was using (and, when using a paid plan, avoids being billed for an index we no longer need).

In [ ]:
# To delete the Pinecone index, 
# uncomment the line below + run this code cell

pc.delete_index(index_name)